# One-second and video evaluation

Binary FACED analysis using previously stored predictions; no retraining.

In [ ]:
import json,hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,balanced_accuracy_score,f1_score
root=Path(r'D:\EEG Experiment Artifacts\2026-11-02')
out=root/'signal_analysis';out.mkdir(parents=True,exist_ok=True)
def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for part in iter(lambda:f.read(1024*1024),b''):h.update(part)
    return h.hexdigest()
def measure(y,p):
    y=np.asarray(y).reshape(-1);p=np.asarray(p).reshape(-1)
    assert y.shape==p.shape and np.isin(y,[0,1]).all() and np.isin(p,[0,1]).all()
    return {'accuracy':float(accuracy_score(y,p)),'balanced_accuracy':float(balanced_accuracy_score(y,p)),
            'macro_f1':float(f1_score(y,p,average='macro',zero_division=0))}


In [ ]:
# Compare second and video decisions without retraining.
# Majority ties use the mean decision value (or mean logit difference); an exact zero maps to class 0.
rows=[];source_hashes={};predictions={}
def add_rows(model,protocol,pred,y,score,subjects):
    assert pred.shape==y.shape==score.shape and pred.shape[-1]==30
    assert np.array_equal(y,np.broadcast_to(y[...,0,None],y.shape))
    video_y=y[...,0];mean_score=score.mean(axis=-1);votes=pred.sum(axis=-1)
    majority=np.where(votes>15,1,np.where(votes<15,0,(mean_score>0).astype(int)))
    mean_decision=(mean_score>0).astype(int)
    for level,p in [('one_second',pred),('video_majority',majority),('video_mean_score',mean_decision)]:
        rows.append({'model':model,'protocol':protocol,'level':level,'subjects':len(np.unique(subjects)),
                     'videos':int(video_y.size),'predictions':int(p.size),
                     'ties':int((votes==15).sum()) if level=='video_majority' else 0,**measure(y if level=='one_second' else video_y,p)})
    predictions[model+'_video_labels']=video_y;predictions[model+'_video_majority']=majority;predictions[model+'_video_mean_score']=mean_decision
for feature in ['de','psd','de_psd']:
    folder=root/'svm_feature_ablation'/feature
    path=folder/'out_of_fold_predictions.npz';summary=folder/'summary.json'
    saved=np.load(path);pred,y,score=saved['predictions'],saved['labels'],saved['decision']
    assert pred.shape==y.shape==score.shape==(123,24,30)
    assert np.isclose(measure(y,pred)['accuracy'],json.loads(summary.read_text())['accuracy'])
    add_rows('SVM_'+feature,'10-fold subject-disjoint out-of-fold test',pred,y,score,np.arange(123))
    source_hashes[str(path)]=sha256(path);source_hashes[str(summary)]=sha256(summary)
for name,run_id in [('GAT_mean','A00001'),('GAT_max','A00026'),('GAT_attention','A00027'),('GAT_hierarchical','A00028')]:
    folder=root/'graph_experiments'/'runs'/run_id
    path=folder/'validation_predictions.npz';info=folder/'metrics.json';saved=np.load(path)
    pred,y,logits=saved['predictions'],saved['labels'],saved['logits']
    assert pred.shape==y.shape==(528,30) and logits.shape==(528,30,2)
    assert np.isclose(measure(y,pred)['accuracy'],json.loads(info.read_text())['validation']['accuracy'])
    add_rows(name,'single subject-disjoint validation split; checkpoint selected on this validation',pred,y,logits[...,1]-logits[...,0],saved['subjects'])
    source_hashes[str(path)]=sha256(path);source_hashes[str(info)]=sha256(info)
table=pd.DataFrame(rows);table.to_csv(out/'second_vs_video.csv',index=False)
np.savez_compressed(out/'video_predictions.npz',**predictions)
summary={'comparison':'same stored predictions at one-second and per-video levels',
         'tie_rule':'majority ties use sign of mean decision/logit difference; zero maps to class 0',
         'warning':'SVM out-of-fold test and graph validation protocols differ and must not be compared as equivalent tests; stimulus labels are shared across subjects',
         'sources_sha256':source_hashes,'results':rows}
(out/'second_vs_video.json').write_text(json.dumps(summary,indent=2),encoding='utf8')
for protocol,subset in table.groupby('protocol',sort=False):
    pivot=subset.pivot(index='model',columns='level',values='balanced_accuracy')
    ax=pivot.plot.bar(rot=20,ylim=(0.45,0.75),title=protocol);ax.set_ylabel('Balanced accuracy');ax.figure.tight_layout()
    name='svm' if protocol.startswith('10-fold') else 'graph'
    ax.figure.savefig(out/('second_vs_video_'+name+'.png'),dpi=150)
plt.show();print(table.to_string(index=False));print('Saved to:',out)
